# Stickler: Ultra Quick Start

**Already have a Pydantic model? You can evaluate with it as-is, with no configuration required.**

If you have a **Pydantic** model (like the `response_model` your agent produces structured output with) you do **not** need to define a `StructuredModel`, pick comparators, choose thresholds, or write a schema. Hand Stickler your two objects and it picks sensible defaults for you, then shows you exactly what it chose.

```python
import stickler
result = stickler.evaluate(ground_truth, prediction)
```

That's the whole integration. Let's see it.

In [ ]:
# If needed:  pip install stickler-eval
import stickler
print('stickler', stickler.__version__)

## 1. Your model (define nothing extra)

A realistic invoice with the types that usually make evaluation annoying: an **enum**, a **date**, an **optional** field, and a **nested list**. You configure none of it.

In [ ]:
import datetime
import enum
from typing import List, Optional

from pydantic import BaseModel


class Priority(str, enum.Enum):
    LOW = "low"
    HIGH = "high"


class LineItem(BaseModel):
    sku: str
    description: str
    quantity: int
    unit_price: float


class Invoice(BaseModel):
    invoice_id: str
    vendor_name: str
    invoice_date: datetime.date
    total_amount: float
    priority: Priority
    notes: Optional[str] = None
    line_items: List[LineItem] = []

## 2. Two objects you already have

`ground_truth` is what you expected; `prediction` is what your system produced. Note the realistic differences: an abbreviated vendor name, reworded notes, and a **reordered** line-item list.

In [ ]:
ground_truth = Invoice(
    invoice_id="INV-2024-0042",
    vendor_name="Acme Corporation",
    invoice_date=datetime.date(2024, 3, 15),
    total_amount=1247.50,
    priority=Priority.HIGH,
    notes="Net 30 payment terms",
    line_items=[
        LineItem(sku="WM-100", description="Wireless Mouse", quantity=2, unit_price=29.99),
        LineItem(sku="UC-050", description="USB-C Cable 1m", quantity=5, unit_price=12.99),
    ],
)

prediction = Invoice(
    invoice_id="INV-2024-0042",
    vendor_name="Acme Corp",                       # abbreviated
    invoice_date=datetime.date(2024, 3, 15),
    total_amount=1247.50,
    priority=Priority.HIGH,
    notes="net 30 terms",                          # reworded
    line_items=[
        LineItem(sku="UC-050", description="USB-C Cable 1 m", quantity=5, unit_price=12.99),
        LineItem(sku="WM-100", description="Wireless Mouse", quantity=2, unit_price=29.99),
    ],
)

## 3. Evaluate — one line


In [ ]:
result = stickler.evaluate(ground_truth, prediction)

print(f"Overall score: {result.overall_score:.3f}")
print(f"Precision:     {result.precision:.3f}")
print(f"Recall:        {result.recall:.3f}")
print(f"F1:            {result.f1:.3f}")
print()
for field, score in result.field_scores.items():
    print(f"  {field:14} {score:.3f}")

### What just happened (with zero configuration)

- `invoice_id` — matched **exactly** (an ID typo is never "close enough").
- `total_amount` — matched despite being a float, using a small numeric tolerance.
- `invoice_date` — matched as a real **date**, not string edit-distance.
- `priority` — the **enum** matched exactly.
- `notes` — "Net 30 payment terms" vs "net 30 terms" matched with **fuzzy** text matching.
- `line_items` — near-perfect even though the list was **reordered** (Hungarian matching pairs elements optimally, ignoring order).
- `vendor_name` — scored **0.0**. That's the interesting one — see below.

## 4. "But how do I defend the score?"

You don't have to remember or justify anything. `.explain()` tells you exactly what Stickler decided and where each decision came from.

In [ ]:
for field, info in result.explain().items():
    print(f"{field:14} {info['comparator']:24} threshold={info['threshold']:<5} src={info['source']}")

So `vendor_name` scored 0.0 because "Acme Corporation" vs "Acme Corp" fell **below** the `0.85` similarity threshold Stickler picked for a name field — and by default, scores under the threshold are clipped to zero. That is now a one-sentence decision you can defend, or override in one line.

The `src` column explains the *origin* of each choice:

- **`type`** — inferred from the Python type (e.g. `priority: Priority` is an enum → `ExactComparator`).
- **`name-token`** — sharpened by the field name (`*_id` → exact; `*amount` → numeric; `notes` → fuzzy).

## 5. The two knobs (still optional)

Stay lazy, but if you want a little control **without** defining a `StructuredModel`:

- **Reuse the compiled evaluator** across a dataset (faster).
- **`weight_hints=True`** lets field names hint at business importance (id/amount weigh more).

In [ ]:
# Compile once, reuse per pair:
spec = stickler.eval_for(Invoice)
print("reused overall:", round(spec.evaluate(ground_truth, prediction).overall_score, 3))

# Let field names hint at business importance:
weighted = stickler.evaluate(ground_truth, prediction, weight_hints=True)
print("with weight_hints:", round(weighted.overall_score, 3))

# See what the weights became:
for field, info in weighted.explain().items():
    print(f"  {field:14} weight={info['weight']}")

By default every field weighs the same (`weight=1.0`) — business-criticality isn't something a plain model encodes, and we'd rather not guess it silently. `weight_hints=True` turns on name-based weighting, and `.explain()` always shows you exactly what changed.

---

### Next steps

- **Evaluating a Strands agent?** See the *Evaluating a Strands Agent* guide.
- **Scoring a whole test set?** See *Bulk Evaluation*.
- **Want to tune comparators by hand?** Graduate to a hand-authored `StructuredModel` — `stickler.evaluate` is the on-ramp, not a ceiling.